In [6]:
import pandas as pd
import numpy as np

In [2]:
data = 'https://raw.githubusercontent.com/alexeygrigorev/datasets/master/course_lead_scoring.csv'

In [3]:
!wget $data 

--2025-10-13 08:49:47--  https://raw.githubusercontent.com/alexeygrigorev/datasets/master/course_lead_scoring.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 80876 (79K) [text/plain]
Saving to: ‘course_lead_scoring.csv’

course_lead_scoring 100%[===================>]  78.98K  --.-KB/s    in 0.003s  

2025-10-13 08:49:47 (27.0 MB/s) - ‘course_lead_scoring.csv’ saved [80876/80876]



In [8]:
df = pd.read_csv('course_lead_scoring.csv')

In [9]:
df.head()

,lead_source,industry,number_of_courses_viewed,annual_income,employment_status,location,interaction_count,lead_score,converted
0,paid_ads,NaN,1,79450.0,unemployed,south_america,4,0.94,1
1,social_media,retail,1,46992.0,employed,south_america,1,0.80,0
2,events,healthcare,5,78796.0,unemployed,australia,3,0.69,1
3,paid_ads,retail,2,83843.0,NaN,australia,1,0.87,0
4,referral,education,3,85012.0,self_employed,europe,3,0.62,1


In [10]:
df.isnull().sum()

lead_source                 128
industry                    134
number_of_courses_viewed      0
annual_income               181
employment_status           100
location                     63
interaction_count             0
lead_score                    0
converted                     0
dtype: int64

In [11]:
df.dtypes

lead_source                  object
industry                     object
number_of_courses_viewed      int64
annual_income               float64
employment_status            object
location                     object
interaction_count             int64
lead_score                  float64
converted                     int64
dtype: object

In [12]:
df.annual_income = df.annual_income.fillna(0)

In [13]:
df.lead_source = df.lead_source.fillna('NA')
df.industry = df.industry.fillna('NA')
df.employment_status = df.employment_status.fillna('NA')
df.location = df.location.fillna('NA')

In [15]:
df.isnull().sum()

lead_source                 0
industry                    0
number_of_courses_viewed    0
annual_income               0
employment_status           0
location                    0
interaction_count           0
lead_score                  0
converted                   0
dtype: int64

In [16]:
df.industry.value_counts().index[0]

'retail'

In [18]:
numerical = ['number_of_courses_viewed','annual_income','interaction_count','lead_score','converted']
df_num = df[numerical]

In [20]:
correlation_matrix = df_num.corr()
print("Correlation Matrix:\n", correlation_matrix)

Correlation Matrix:
                           number_of_courses_viewed  annual_income  \
number_of_courses_viewed                  1.000000       0.009770   
annual_income                             0.009770       1.000000   
interaction_count                        -0.023565       0.027036   
lead_score                               -0.004879       0.015610   
converted                                 0.435914       0.053131   

                          interaction_count  lead_score  converted  
number_of_courses_viewed          -0.023565   -0.004879   0.435914  
annual_income                      0.027036    0.015610   0.053131  
interaction_count                  1.000000    0.009888   0.374573  
lead_score                         0.009888    1.000000   0.193673  
converted                          0.374573    0.193673   1.000000  


In [21]:
from sklearn.model_selection import train_test_split

In [22]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=42)

In [23]:
len(df_train), len(df_val), len(df_test)

(876, 293, 293)

In [24]:
df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

In [25]:
y_train = df_train.converted.values
y_val = df_val.converted.values
y_test = df_test.converted.values

del df_train['converted']
del df_val['converted']
del df_test['converted']

In [26]:
from sklearn.metrics import mutual_info_score

In [27]:
def mutual_info_conv_score(series):
    return mutual_info_score(series, df_full_train.converted)

In [75]:
categorical = ['lead_source','industry','employment_status','location']

In [76]:
mi = df_full_train[categorical].apply(mutual_info_conv_score)
mi.sort_values(ascending=False).round(4)

lead_source          0.0257
employment_status    0.0133
industry             0.0117
location             0.0023
dtype: float64

In [31]:
from sklearn.feature_extraction import DictVectorizer

In [33]:
numerical = ['number_of_courses_viewed','annual_income','interaction_count','lead_score']

In [34]:
dv = DictVectorizer(sparse=False)

train_dict = df_train[categorical + numerical].to_dict(orient='records')
X_train = dv.fit_transform(train_dict)

val_dict = df_val[categorical + numerical].to_dict(orient='records')
X_val = dv.transform(val_dict)

In [35]:
from sklearn.linear_model import LogisticRegression
model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
model.fit(X_train, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,42
,solver,'liblinear'
,max_iter,1000
,multi_class,'deprecated'


In [36]:
y_pred = model.predict_proba(X_val)[:, 1]

In [37]:
churn_decision = (y_pred >= 0.5)

In [44]:
accuracy_full = (y_val == churn_decision).mean()
accuracy_full.round(2)

np.float64(0.7)

In [45]:
categorical = ['industry','employment_status','location']

In [46]:
dv = DictVectorizer(sparse=False)

train_dict = df_train[categorical + numerical].to_dict(orient='records')
X_train = dv.fit_transform(train_dict)

val_dict = df_val[categorical + numerical].to_dict(orient='records')
X_val = dv.transform(val_dict)

In [47]:
model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
model.fit(X_train, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,42
,solver,'liblinear'
,max_iter,1000
,multi_class,'deprecated'


In [48]:
y_pred = model.predict_proba(X_val)[:, 1]

In [49]:
churn_decision = (y_pred >= 0.5)

In [50]:
accuracy_no_lead = (y_val == churn_decision).mean()
accuracy_no_lead

np.float64(0.7030716723549488)

In [51]:
accuracy_no_lead_diff = accuracy_full - accuracy_no_lead
accuracy_no_lead_diff

np.float64(-0.0034129692832765013)

In [52]:
categorical = ['lead_source','employment_status','location']

In [53]:
dv = DictVectorizer(sparse=False)

train_dict = df_train[categorical + numerical].to_dict(orient='records')
X_train = dv.fit_transform(train_dict)

val_dict = df_val[categorical + numerical].to_dict(orient='records')
X_val = dv.transform(val_dict)

In [54]:
model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
model.fit(X_train, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,42
,solver,'liblinear'
,max_iter,1000
,multi_class,'deprecated'


In [55]:
y_pred = model.predict_proba(X_val)[:, 1]

In [56]:
churn_decision = (y_pred >= 0.5)

In [57]:
accuracy_no_ind = (y_val == churn_decision).mean()
accuracy_no_ind

np.float64(0.6996587030716723)

In [60]:
accuracy_no_ind_diff = accuracy_full - accuracy_no_ind
accuracy_no_ind_diff

np.float64(0.0)

In [61]:
categorical = ['lead_source','industry','location']

In [62]:
dv = DictVectorizer(sparse=False)

train_dict = df_train[categorical + numerical].to_dict(orient='records')
X_train = dv.fit_transform(train_dict)

val_dict = df_val[categorical + numerical].to_dict(orient='records')
X_val = dv.transform(val_dict)

In [63]:
model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
model.fit(X_train, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,42
,solver,'liblinear'
,max_iter,1000
,multi_class,'deprecated'


In [64]:
y_pred = model.predict_proba(X_val)[:, 1]

In [65]:
churn_decision = (y_pred >= 0.5)

In [66]:
accuracy_no_emp = (y_val == churn_decision).mean()
accuracy_no_emp

np.float64(0.6962457337883959)

In [68]:
accuracy_no_emp_diff = accuracy_full - accuracy_no_emp
accuracy_no_emp_diff

np.float64(0.0034129692832763903)

In [86]:
categorical = ['lead_source','industry','employment_status','location']

dv = DictVectorizer(sparse=False)

train_dict = df_train[categorical + numerical].to_dict(orient='records')
X_train = dv.fit_transform(train_dict)

val_dict = df_val[categorical + numerical].to_dict(orient='records')
X_val = dv.transform(val_dict)

In [87]:
for C in [0.01, 0.1, 1, 10, 100]:

    model = LogisticRegression(solver='liblinear', C=C, max_iter=1000, random_state=42)
    model.fit(X_train, y_train)

    y_pred = model.predict_proba(X_val)[:, 1]
    churn_decision = (y_pred >= 0.5)
    accuracy = (y_val == churn_decision).mean()
    
    print(C, accuracy)

0.01 0.6996587030716723
0.1 0.6996587030716723
1 0.6996587030716723
10 0.6996587030716723
100 0.6996587030716723
